# Bài tập - Chọn đúng bộ lọc cho đúng loại nhiễu
**BVU - Cao học - Xử lý ảnh - Trương Đình Phúc**

Cùng một ảnh, thêm 2 loại nhiễu: **nhiễu Gaussian** (hạt mịn, giống nhiễu cảm biến) và **nhiễu muối tiêu** (chấm đen trắng, nhiễu xung). Áp cả `GaussianBlur` và `medianBlur` lên từng ảnh, đo **PSNR** so với ảnh gốc sạch (PSNR càng cao càng gần ảnh gốc = lọc càng tốt), để rút ra bộ lọc nào phù hợp với loại nhiễu nào.

**Đoán trước khi chạy:** bộ lọc nào sẽ thắng ở nhiễu Gaussian? Ở nhiễu muối tiêu?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

FOLDER = "/content/drive/MyDrive/Colab Notebooks"

def load(name):
    extensions = [".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"]
    for ext in extensions:
        path = os.path.join(FOLDER, name + ext)
        if os.path.exists(path):
            img = cv2.imread(path)
            if img is not None:
                print("Đã đọc ảnh:", path)
                return img
    raise FileNotFoundError(f"Không tìm thấy ảnh '{name}' trong thư mục {FOLDER}")

def show(items, title, gray=False):
    n = len(items)
    plt.figure(figsize=(4 * n, 4))
    for i, (name, img) in enumerate(items, start=1):
        plt.subplot(1, n, i)
        plt.title(name)
        plt.axis("off")
        if gray or img.ndim == 2:
            plt.imshow(img, cmap="gray", vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## Tạo 2 loại nhiễu trên ảnh gốc

In [ ]:
goc = cv2.cvtColor(load("house"), cv2.COLOR_BGR2GRAY)  # anh goc sac net, chuyen xam

rng = np.random.default_rng(0)

# Nhieu Gaussian: cong nhieu ngau nhien phan phoi chuan vao tung pixel
gaussian = np.clip(goc + rng.normal(0, 20, goc.shape), 0, 255).astype(np.uint8)

# Nhieu muoi tieu: mot ti le pixel bi ep thanh 0 (tieu) hoac 255 (muoi)
muoitieu = goc.copy()
m = rng.random(goc.shape)
muoitieu[m < 0.05] = 0
muoitieu[m > 0.95] = 255

show([("Goc", goc), ("Nhieu Gaussian", gaussian), ("Nhieu muoi tieu", muoitieu)],
     "Anh goc va 2 loai nhieu", gray=True)

## Áp 2 bộ lọc (GaussianBlur, medianBlur) và đo PSNR

In [ ]:
def psnr(sach, loc):
    mse = np.mean((sach.astype(float) - loc.astype(float)) ** 2)
    return 99 if mse == 0 else 10 * np.log10(255 ** 2 / mse)

ksize = 5  # kich thuoc cua so loc (so LE: 3, 5, 7)

print("%-16s %9s %9s %9s" % ("Loai nhieu", "Nhieu", "Gaussian", "Median"))
ket_qua = {}
for ten, anh in [("Nhieu Gaussian", gaussian), ("Nhieu muoi tieu", muoitieu)]:
    loc_g = cv2.GaussianBlur(anh, (ksize, ksize), 0)
    loc_med = cv2.medianBlur(anh, ksize)
    ket_qua[ten] = (psnr(goc, anh), psnr(goc, loc_g), psnr(goc, loc_med))
    print("%-16s %9.1f %9.1f %9.1f" % (ten, *ket_qua[ten]))

In [ ]:
show([
    ("Muoi tieu", muoitieu),
    ("GaussianBlur", cv2.GaussianBlur(muoitieu, (ksize, ksize), 0)),
    ("medianBlur", cv2.medianBlur(muoitieu, ksize))
], "So sanh 2 bo loc tren nhieu muoi tieu", gray=True)

show([
    ("Nhieu Gaussian", gaussian),
    ("GaussianBlur", cv2.GaussianBlur(gaussian, (ksize, ksize), 0)),
    ("medianBlur", cv2.medianBlur(gaussian, ksize))
], "So sanh 2 bo loc tren nhieu Gaussian", gray=True)

## Nhận xét

- **PSNR** (Peak Signal-to-Noise Ratio) đo mức chênh lệch giữa ảnh đã lọc và ảnh gốc sạch: PSNR càng cao thì ảnh sau lọc càng gần ảnh gốc (lọc càng hiệu quả).
- **Nhiễu Gaussian** phân bố đều, biên độ nhỏ trên toàn ảnh -> **GaussianBlur** (lấy trung bình có trọng số) phù hợp hơn vì làm mượt được nhiễu dạng liên tục.
- **Nhiễu muối tiêu** là các điểm outlier cực trị (0 hoặc 255) rời rạc -> **medianBlur** (lấy trung vị) phù hợp hơn hẳn, vì trung vị của 1 cửa sổ chứa vài điểm outlier vẫn giữ nguyên giá trị đúng, trong khi trung bình (Gaussian) bị outlier kéo lệch.

**Kết luận:** phải nhìn **loại nhiễu** trước khi chọn bộ lọc, không chọn theo thói quen: nhiễu cảm biến (Gaussian) → `GaussianBlur`; nhiễu xung (muối tiêu) → `medianBlur` mới khử sạch mà vẫn giữ được biên ảnh.